# German OCR and Text-to-speech

## 🚀 Setup Guide for Other Users

To run this backend successfully, you will need to configure your own **ngrok** authentication token. This allows ngrok to expose your Flask server running inside this Colab notebook to a secure, public URL.

### 🔑 Step-by-Step Ngrok Configuration:
1. **Get an Auth Token**: Sign up or log in to your account at [ngrok.com](https://ngrok.com/) and copy your personal **Authtoken** from your dashboard.
2. **Store it in Colab Secrets**:
   * Click on the **Secrets (key icon 🔑)** on the left-hand sidebar of Google Colab.
   * Add a new secret with the name: `NGROK_AUTH_TOKEN`
   * Paste your copied ngrok token as the value.
   * Toggle the switch next to it to grant **Notebook access** to this secret.
3. **Run the Notebook**: Execute all cells in order. The system will automatically pick up your secret token without exposing it in the code.

1. Install paddleOCR (seperately)

In [ ]:
!pip install --force-reinstall "paddleocr<3.0" paddlepaddle
!apt-get install libgl1 -qq > /dev/null

  Using cached paddleocr-2.10.0-py3-none-any.whl.metadata (12 kB)
  Using cached paddlepaddle-3.3.1-cp313-cp313-manylinux1_x86_64.whl.metadata (8.8 kB)
  Using cached shapely-2.2.0-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (6.7 kB)
  Using cached scikit_image-0.26.0-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (15 kB)
  Using cached pyclipper-1.4.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (8.6 kB)
  Using cached lmdb-3.0.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (1.2 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached numpy-2.5.4-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached rapidfuzz-3.14.6-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (12 kB)
  Using cached opencv_python-5.0.0.93-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached opencv_contrib_

2. Verify PaddleOCR Installation

In [ ]:
!pip list | grep -E "paddleocr|paddlepaddle"

paddleocr                             2.10.0
paddlepaddle                          3.3.1


3. Generate Ngrok URL (Re-do everytime running the system)

In [5]:
!pip install pyngrok --upgrade --quiet
from pyngrok import ngrok
from google.colab import userdata

# Retrieve the token from your Colab Secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(5000)
print("🔥 Your ngrok URL is:", public_url)       # Manually expose port 5000

🔥 Your ngrok URL is: NgrokTunnel: "https://f354-34-80-169-35.ngrok-free.app" -> "http://localhost:5000"


4. Run Back-end System

In [3]:
# ====== INSTALLATIONS ======
!pip install flask-ngrok flask-cors pyngrok gtts torch transformers opencv-python-headless numpy --quiet
!pip install "paddleocr<3.0" --quiet
!pip install pyngrok --upgrade --quiet

# ====== IMPORTS & SETUP ======
from flask import Flask, request, send_file, jsonify
from flask_ngrok import run_with_ngrok
from flask_cors import CORS
from paddleocr import PaddleOCR
from gtts import gTTS
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import cv2
import numpy as np
import io
import logging
from pyngrok import ngrok, conf

# Initialize logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ====== MODEL LOADING ======
# OCR Model
ocr = PaddleOCR(
    use_angle_cls=True,
    lang='german',
    show_log=False,
    use_gpu=True if torch.cuda.is_available() else False
)

# Initialize Flask
app = Flask(__name__)
CORS(app)
run_with_ngrok(app)

@app.route('/extract-text', methods=['POST'])
def extract_text():
    if 'image' not in request.files:
        return jsonify({"error": "No image provided"}), 400

    try:
        # 1. Read uploaded image
        img_bytes = request.files['image'].read()

        # 2. Convert to OpenCV format
        nparr = np.frombuffer(img_bytes, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        # 3. Extract text with PaddleOCR
        cv2.imwrite("debug_image.jpg", img) #troubleshooting
        result = ocr.ocr(img, cls=True)
        text = ' '.join([line[1][0] for line in result[0]]) if result else ""

        if not text:
            return jsonify({"error": "No text detected in image"}), 400

        return jsonify({
            "status": "success",
            "text": text
        })

    except Exception as e:
        logger.error(f"Text extraction failed: {str(e)}")
        return jsonify({"error": str(e)}), 500

@app.route('/generate-audio', methods=['POST'])
def generate_audio():
    if 'text' not in request.json:
        return jsonify({"error": "No text provided"}), 400

    try:
        text = request.json['text']

        # Convert to German audio
        audio = io.BytesIO()
        gTTS(text=text, lang='de').write_to_fp(audio)
        audio.seek(0)

        return send_file(
            audio,
            mimetype='audio/mpeg',
            as_attachment=True,
            download_name='german_audio.mp3'
        )

    except Exception as e:
        logger.error(f"Audio generation failed: {str(e)}")
        return jsonify({"error": str(e)}), 500

print("✅ API Ready! Two endpoints available:")
print("1. POST /extract-text - Returns extracted text")
print("2. POST /generate-audio - Returns audio file")
app.run()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.2/544.2 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 52.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 1.16.1 which is incompatible.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


download https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_det_infer.tar to /root/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer/en_PP-OCRv3_det_infer.tar


100%|██████████| 3910/3910 [00:11<00:00, 343.62it/s] 


download https://paddleocr.bj.bcebos.com/PP-OCRv3/multilingual/latin_PP-OCRv3_rec_infer.tar to /root/.paddleocr/whl/rec/latin/latin_PP-OCRv3_rec_infer/latin_PP-OCRv3_rec_infer.tar


...

100%|██████████| 2138/2138 [00:03<00:00, 571.50it/s]


✅ API Ready! Two endpoints available:
1. POST /extract-text - Returns extracted text
2. POST /generate-audio - Returns audio file
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


 * Running on http://d898-34-80-169-35.ngrok-free.app
 * Traffic stats available on http://127.0.0.1:4040


INFO:werkzeug:127.0.0.1 - - [16/Oct/2026 02:56:51] "POST /extract-text HTTP/1.1" 200 -
ERROR:__main__:Text extraction failed: could not execute a primitive
INFO:werkzeug:127.0.0.1 - - [16/Oct/2026 02:57:44] "POST /extract-text HTTP/1.1" 500 -
ERROR:__main__:Text extraction failed: could not execute a primitive
INFO:werkzeug:127.0.0.1 - - [16/Oct/2026 02:58:01] "POST /extract-text HTTP/1.1" 500 -
INFO:werkzeug:127.0.0.1 - - [16/Oct/2026 02:59:42] "POST /extract-text HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [16/Oct/2026 02:59:49] "OPTIONS /generate-audio HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [16/Oct/2026 02:59:51] "POST /generate-audio HTTP/1.1" 200 -
ERROR:root:Unexpected exception finding object shape
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/google/colab/_debugpy_repr.py", line 54, in get_shape
    shape = getattr(obj, 'shape', None)
  File "/usr/local/lib/python3.13/dist-packages/werkzeug/local.py", line 318, in __get__
    obj = inst

## 🔌 How to Integrate this Backend into Your Own Projects

Once this backend is running and you have obtained your public `ngrok` URL (e.g., `https://f354-34-80-169-35.ngrok-free.app`), you can interact with these endpoints from any external frontend or backend application.

### Extract German Text from an Image

**Endpoint:** `POST /extract-text`  
**Content-Type:** `multipart/form-data`  

#### JavaScript / Fetch Example:
```javascript
const formData = new FormData();
formData.append('image', imageFileInputElement.files[0]);

fetch('https://YOUR-NGROK-SUBDOMAIN.ngrok-free.app/extract-text', {
  method: 'POST',
  body: formData
})
.then(response => response.json())
.then(data => console.log(data));
